# Day 2 重考卷｜张量操作与自动微分

> **规则**：闭卷（不看 day2_notes），限时 45–60 分钟；编程题必须跑通再交。
> **本卷说明**：题目与上次**不重复**，但覆盖同样的范围，并把上次 8 个红点（A4、A7、B1、B7、C2、C3、C4、D3）全部换角度重考了一遍。
> **作答方式**：A、B 组写在每节下面的"作答区"里；C、D 组直接改/运行代码。

**考点分布**：线代 2 题 ｜ 三步曲 1 题 ｜ 张量 shape 3 题 ｜ 广播 2 题 ｜ autograd 4 题 ｜ 编程 4 题 ｜ 找 bug 3 题

---

## A 组｜概念（8 题）

**A1.** 一个 2×2 矩阵的两列分别是 (1, 2) 和 (3, 4)。用"基向量跑哪去了"这句话说出这个矩阵做了什么；再说明 `2A` 与 `A` 在几何上差在哪。

**A2.** "先旋转 90° 再横向拉伸 2 倍"和"先横向拉伸 2 倍再旋转 90°"结果一样吗？用这个例子解释为什么 `AB ≠ BA`。

**A3.** 三步曲里，把 MSE 换成交叉熵属于哪一步？这一步是人的工作还是机器的工作？为什么？

**A4.** 判断对错并改正（4 小题）：
① `*` 是矩阵乘法　② `@` 是逐元素相乘　③ `torch.dot` 要求两个输入都是一维向量　④ `torch.cross` 可以用于二维向量

**A5.** `X` 是 (3,4)、`Y` 是 (4,4)：`torch.cat((X, Y), dim=0)` 和 `dim=1` 哪个合法？合法的那一个形状是多少？另一个为什么报错？

**A6.** 训练循环里为什么最终一定要把 loss 变成标量？请从"链式法则的起点"这个角度回答。

**A7.** 下面哪些场景**必须**用 `.detach()`？逐条说明理由：
① 把带梯度的张量转成 numpy 去画图　② 让某个中间结果不参与求导　③ 让某个参数永远不被训练　④ 计算 target network 的目标值

**A8.** 为什么 PyTorch 的梯度默认是累加的？举一个"故意利用累加"的真实用法。

### 作答区 A

```
A1.经过变换后矩阵的基向量分别落在(1, 2)和(3, 4)处; 2A的基向量都是A的两倍

A2.不一样; 这是两种不同的变换，顺序不能互换

A3.定义损失; 人的工作; 人根据具体情况可以更换损失函数

A4.x x ✅ x

A5.dim=0; (7, 4); 维度不匹配

A6.链式法则需要一个起点，而标量时这个起点天然为1

A7.1: 需要获取实时的张量用于画图 2: 表明计算路径到此为止 3: 用requires_grad=False

A8.多个loss/分支的梯度可以直接相加; 多任务训练
```


## B 组｜计算（8 题，每题都必须写出形状）

**B1.** `X = torch.arange(24).reshape(2, 3, 4)`：写出 `X[0]`、`X[0,1]`、`X[:,1,:]`、`X[...,0]` 的形状。

**B2.** `x = torch.tensor([1.0, 2.0, 3.0])`：写出 `x.shape`、`x[1].shape`、`x[1:2].shape`，并解释后两个为什么不一样。

**B3.** `x = torch.tensor([1.0, 2.0, 3.0])`、`y = (x*x).sum()`：写出 backward 之后 `x.grad` 的值（写成向量的形式）；再写出**不清零**跑第 2、3 次之后的值；最后给出跑 k 次的通式。

**B4.** `x = torch.tensor([1.0, 2.0, 3.0])`、`u = (x*x).detach()`、`z = (u*x).sum()`：求 ∂z/∂x 的数值；并说明如果**去掉** detach，结果会变成什么。

**B5.** `X` 是 (8,5)、`w` 是 (5,1)：`(X @ w).shape` 是多少？如果写成 `X @ w.reshape(-1)` 呢？再分别与 `(8,1)` 的标签相减，两种写法各会发生什么？

**B6.** `a = torch.ones(4, 1)`、`b = torch.ones(1, 5)`：`(a + b).shape` 是多少？`(a + b).sum()` 是多少？

**B7.** `x = torch.tensor([2.0, 4.0])`：写出 `x ** 2`、`2 ** x`、`torch.exp(x)` 的数值。

**B8.** `x = torch.tensor([1.0, 2.0, 3.0])`、`y = x * x`：分别写出 `y.sum().backward()` 和 `y.backward(torch.tensor([1.0, 0.0, 0.0]))` 之后的 `x.grad`；并解释第二种写法在算什么。

### 作答区 B

```
B1. 

B2.(3,); (1,); (1, 1); 一个有1个纬度，一个有2个纬度

B3.[2, 4, 6]; [4, 8, 12]; [6, 12, 18]; [2k, 4k, 6k]

B4.[1, 4, 9]; [3, 12, 27]

B5.(8, 1); (8,); (8, 8)

B6.(4, 5); 20

B7.[4, 16]; [4, 16]; [e^2, e^4]

B8.[2, 4, 6]; [2, 0, 0]; 第一个分量的梯度
```


## C 组｜编程（4 题，完成后必须跑通）

**C1.** 创建 `x = torch.arange(12)`，reshape 成 (3,4)，然后打印三个形状：`x.shape`、reshape 结果的 shape、`x.view(-1).shape`。最后用一句话回答：reshape 有没有改变 x 本身？

**C2.** 造一个 (50, 4) 的随机张量，按列做 **min-max 归一化**（每列缩放到 [0,1]），不许用 for 循环。打印每列的 min 和 max 做验证。

**C3.** 写函数 `g(a)`：内部必须含 `for` 循环和 `if/else`，返回一个标量。用 `requires_grad=True` 的标量输入，backward 之后把你算出的梯度和手推值做对比，并写出你的手推值是多少。

**C4.** 写一个最小训练循环（自造数据），要求：迭代前清零、打印 loss、loss 确实下降。然后做对照实验：把清零那一行注释掉再跑一次，说明发生了什么、为什么。


In [ ]:
# C1
import torch

x = torch.arange(12)
x2 = x.reshape(3, 4)

# TODO: 打印 x.shape、x2.shape、x.view(-1).shape
print(x.shape, x2.shape, x.view(-1).shape)
# 一句话回答（写在注释里）：reshape 有没有改变 x 本身？
# 没有


torch.Size([12]) torch.Size([3, 4]) torch.Size([12])


In [3]:
# C2
import torch

torch.manual_seed(0)
x = torch.randn(50, 4)

# TODO: 按列做 min-max 归一化（不许用 for）
# 提示：x.min(dim=0).values 的形状是 (4,)，可以广播

min_vals = x.min(dim=0).values
max_vals = x.max(dim=0).values
xn = (x - min_vals) / (min_vals - max_vals)

# TODO: 打印每列的 min 和 max 验证
print("每列的最小值:", xn.min(dim=0).values)
print("每列的最大值:", xn.max(dim=0).values)


每列的最小值: tensor([-1., -1., -1., -1.])
每列的最大值: tensor([-0., -0., -0., -0.])


In [17]:
# C3
import torch

def g(a):
    # TODO: 内部含 for 循环 + if/else，返回标量
    for i in range(10):
        if a > 0:
            a = a * 2
        else:
            a = a - 1
    return a

a = torch.randn(size=(), requires_grad=True)
# TODO: d = g(a); d.backward(); 打印 a.grad
d = g(a)
d.backward()
print(a.grad)
print(d / a)
# TODO: 写出你手推的梯度表达式，并断言两者相等
# 手推梯度表达式：∂d/∂a = 2^10 = 1024 (当 a > 0 时)
# 或者 ∂d/∂a = -1 (当 a <= 0 时)
assert (d / a).item() == 1024 or (d / a).item() == -1


tensor(1024.)
tensor(1024., grad_fn=<DivBackward0>)


In [4]:
# C4
import torch

torch.manual_seed(0)
w = torch.tensor([0.0], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
x = torch.randn(100, 1)
y = 3 * x + 2 + 0.1 * torch.randn(100, 1)

for step in range(5):
    y_hat = x @ w + b
    loss = ((y_hat - y) ** 2).mean()

    # TODO: 清零
    w.grad = None
    b.grad = None
    # TODO: 反向传播
    loss.backward()
    # TODO: 更新参数（with torch.no_grad()）
    with torch.no_grad():
        w -= 0.1 * w.grad
        b -= 0.1 * b.grad
    print(f"step {step} loss={loss.item():.4f}")

# TODO: 把清零那行注释掉再跑一次，在注释里写结论
# 结论：不清零的话，梯度会累积

step 0 loss=13.9028
step 1 loss=12.2725
step 2 loss=11.2303
step 3 loss=10.5639
step 4 loss=10.1377


## D 组｜找 bug（3 题）

每题三步：① 先运行代码观察结果（可能报错，也可能静默算错）；② 指出**错在哪**；③ 给出**修法**，并说明修完之后的正确结果。

### D1


In [7]:
import torch

w = torch.tensor([1.0, 2.0], requires_grad=True)
X = torch.ones(3, 2)

for step in range(2):
    y = X @ w
    loss = ((y - 1) ** 2).sum()
    w.grad = None
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad

print(w)
# 诊断：w的梯度没有清零
# 修法：w.grad=None
# 修完的正确结果：如下


tensor([0.0400, 1.0400], requires_grad=True)


In [ ]:
import torch

x = torch.tensor([1.0, 2.0], requires_grad=True)
y = (x * 3).detach()
z = (y * x).sum()
z.backward()

print(x.grad)      # 作者期望得到 6x
# 诊断：不用detach()就行
# 修法：z = (y * x).sum() 改为 z = (x * 3 * x).sum()
# 修完的正确结果：tensor([6., 12.])


tensor([3., 6.])


In [ ]:
import torch

X = torch.randn(16, 3)
w = torch.randn(3)
y = torch.randn(16, 1)

y_hat = X @ w
loss = ((y_hat - y) ** 2).mean()

print("y_hat.shape =", y_hat.shape)
print("loss =", loss)
# 诊断：w的形状不对，应该是 (3, 1)
# 修法（两种）：w = torch.randn(3, 1) 或者 y_hat = X @ w.unsqueeze(1)


y_hat.shape = torch.Size([16])
loss = tensor(5.9011)


### 作答区 D

```
D1 诊断 / 修法 / 结果：


D2 诊断 / 修法 / 结果：


D3 诊断 / 修法（两种）：

```

---

## 交卷前自检

- [ ] 所有要求"写形状"的地方都写了形状（B 组 8 题每题都有）
- [ ] C 组四个 cell 都跑通了，没有 `None` 残留
- [ ] D 组三个 snippet 都实际运行过，不是"看代码猜"


In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
print(x[0].shape, x[0, 1].shape, x[:, 1, :].shape, x[..., 0].shape)
y = torch.tensor([1., 2., 3.])
print(y.shape, y[1].shape, y[1:2].shape)      # 注意 () 和 (1,) 的区别